# Tutorial 1: Introduction to Statistical Modelling and Planned Peeking for A/B Testing Optimization

## Learning Objectives

After completing this week's worksheet and tutorial work, you will be able to:

1. Discuss why the methods you have learned in past courses are not sufficient to answer the more complex research problems being posed in this course (in particular, stopping an A/B test early).
2. Explain principled peeking and how it can be used for early stopping of an experiment (e.g., A/B testing). 
3. Write a computer script to perform A/B testing optimization using principled peeking.
4. Discuss the limitations of principled peeking for A/B testing optimization/early stopping of experiments.

In [ ]:
# Run this cell before continuing.
library(digest)
library(testthat)
library(tidyverse)
library(infer)
library(broom)
library(cowplot)
library(binom)
source("tests_tutorial_01.R")

In [ ]:
# Two-sample t-test for tracking p-values by incremental sample sizes until getting to n.

# @param n (numeric): Initially planned sample size for each group, n = n_current = n_new.
# @param d_0 (numeric): effect size.
# @param sd_current (numeric): Population standard deviation for current ad.
# @param sd_new (numeric): Population standard deviation for new ad.
# @param sample_increase_step (numeric): Sample size increment.

# @return p.value.df: A tibble that has 2 columns:
# inc_sample_size (increasing sample size) and p_value (p-value from performing a two-sample t-test).

incremental_t_test <- function(n, d_0, sd_current, sd_new) {
  sample_current <- rnorm(n, mean = 4, sd = sd_current)
  sample_new <- rnorm(n, mean = 4 + d_0, sd = sd_new)

  p.value.df <- tibble(
    inc_sample_size = rep(0, 1 + (n-10) / sample_increase_step),
    p_value = rep(0, 1 + (n-10) / sample_increase_step)
  )

  current_sample_size <- 10
  
  for (i in 1:nrow(p.value.df))
  {
    t_test_results <- t.test(sample_new[1:current_sample_size], sample_current[1:current_sample_size],
      var.equal = TRUE,
      alternative = "greater"                      
    )
    p.value.df[i, "p_value"] <- as_tibble(t_test_results$p.value)
    p.value.df[i, "inc_sample_size"] <- current_sample_size
    current_sample_size <- current_sample_size + sample_increase_step
  }

  return(p.value.df)
}

## 1. Inferential Implications of Early Stopping in Hypothesis Testing

In Worksheet 1, we studied by means of an A/A testing (i.e., the case where we know that there's no difference between the groups) how peeking can inflate the probability of Type I Error. There are two main aspects with peeking:
1. The sample size is changing every time you peek;

2. You are conducting multiple statistical analyses (e.g., hypothesis testing, confidence intervals);


**Question 1.1**
<br>{points: 1}

In the context of hypothesis testing, is the following statement true or false? 

> The problem with peeking too early in the study is that we might end up rejecting $H_0$ when we have a small sample size (i.e., the test is not powerful). Therefore we can't trust the rejection of $H_0$. 

_Assign your answer to an object called `answer1.1`. Your answer should be either "true" or "false", surrounded by quotes._

In [ ]:
# answer1.1 <- ...

# your code here
fail() # No Answer - remove if you provide an answer

In [ ]:
# Here we check to see if you have given your answer the correct object name
# and if your answer is plausible. However, all other tests have been hidden
# so you can practice deciding when you have the correct answer.
test_that('Did not assign answer to an object called "answer1.1"', {
  expect_true(exists("answer1.1"))
})
test_that('Answer should be "true" or "false"', {
  expect_match(answer1.1, "true|false", ignore.case = TRUE)
})

**Question 1.2**
<br>{points: 1}

In the context of hypothesis testing, is the following statement true or false? 

> When peeking multiple times, we should use the same significance level each time, so there is a high likelihood that the probability of Type I Error is close to the specified significance level. 

_Assign your answer to an object called `answer1.2`. Your answer should be either "true" or "false", surrounded by quotes._

In [ ]:
# answer1.2 <- ...

# your code here
fail() # No Answer - remove if you provide an answer

In [ ]:
# Here we check to see if you have given your answer the correct object name
# and if your answer is plausible. However, all other tests have been hidden
# so you can practice deciding when you have the correct answer.
test_that('Did not assign answer to an object called "answer1.2"', {
  expect_true(exists("answer1.2"))
})
test_that('Answer should be "true" or "false"', {
  expect_match(answer1.2, "true|false", ignore.case = TRUE)
})

We learned in the Worksheet 1 that multiple hypothesis tests can drastically inflate the probability of Type I Error. However, we learned in STAT 201 that we could account for the fact that our analysis requires multiple tests by adjusting the p-value. There are different strategies that we could use to adjust the p-value (e.g., Bonferroni, BH). Would that be enough, though? In the next few exercises, you are going to investigate that. We will focus on the Bonferroni correction.

The only problem is: to investigate the impacts of p-value adjustments, we need to know the truth! In A/B tests, we can deal with this situation fairly easily: we can conduct an A/A test. An A/A test is the same thing as an A/B test, except that we use the same treatment for both 
groups. This way, we know the truth; there is no difference! 

Although one might be tempted to monitor the test "online" (i.e., they peek every time a new observation comes) so they can detect a difference as soon as possible, this would require too many hypothesis tests. However, this would require quite heavy adjustments of the p-values.

**Question 1.3**
<br>{points: 1}

Since we decided to use the Bonferroni correction to adjust the p-values, 
one might be tempted to monitor the test "online" (i.e., they peek every time a new observation comes) so they can detect a difference as soon as possible. Briefly, explain why/why not this is a problem. 

DOUBLE CLICK TO EDIT **THIS CELL** AND REPLACE THIS TEXT WITH YOUR ANSWER.

Having learned about the problems of online monitoring, the TikTok team decided to peek fewer times. They decided to peek for n=35,  60, 110, 260, 960. But they want to investigate if this will actually improve things, so they decided to conduct 2,000 A/A tests to investigate.  We have the data for you!

In [ ]:
tiktok_many_AA_tests <- 
    read_csv("data//tiktok_data.csv")

head(tiktok_many_AA_tests, 10)

**Question 1.4** 
<br> {points: 1}

Considering the new strategy of the TikTok team, what would be the new significance level? Estimate it from the 2000 A/A experiments they conducted by calculating the proportion of experiments that would have wrongly rejected $H_0$ at a 5% significance level.

_Assign your answer to an object called `answer1.4`. Your answer should be a single number._

In [ ]:
# answer1.4 <- ...

# your code here
fail() # No Answer - remove if you provide an answer

answer1.4

In [ ]:
test_1.4()

**Question 1.5** 
<br> {points: 1}

Add a new column to `tiktok_many_AA_tests` with the adjusted p-values using the Bonferroni method. The column should be named `adjusted_pvalue`.

In [ ]:
# tiktok_many_AA_tests <- ...

# your code here
fail() # No Answer - remove if you provide an answer

head(tiktok_many_AA_tests,12)

In [ ]:
test_1.5()

**Question 1.6** 
<br> {points: 1}

Considering the new strategy of the TikTok team, what would be the new significance level? Estimate it from the 2000 A/A experiments they conducted by calculating the proportion of experiments that would have wrongly rejected $H_0$ at a 5% significance level.

_Assign your answer to an object called `answer1.6`. Your answer should be a single number._

In [ ]:
# answer1.6 <- ...

# your code here
fail() # No Answer - remove if you provide an answer

answer1.6

In [ ]:
test_1.6()

As you can see, by choosing a reasonable number of peeks, and adjusting the p-values, we can make a sound statistical analysis. This is called *principled peeking*. 

## 2. Analysis of an A/B Testing Paper

In this exercise, we will work with the paper ["Improving Library User Experience with A/B Testing: Principles and Process"](https://quod.lib.umich.edu/w/weave/12535642.0001.101?view=text;rgn=main) by Young (2014). This paper presents a case study where A/B testing is applied with different webpage designs. The primary aim is to compare user interactions to determine which one statistically improves the navigation experience by increasing the homepage click-through rate. The experiment was conducted using the web analytics software Google Analytics and Crazy Egg. The data from the paper can be found [here](https://scholarworks.montana.edu/xmlui/handle/1/3507).

The setup was done on the **Interact** category in the Montana State University's library webpage (more information can be found in the section *Step 1* in the paper). The experimental treatments (as explained and shown in *Step 4* in the paper) are the following: **Interact** (the control treatment), **Connect**, **Learn**, **Help**, and **Services**. The response variable is what we call the **click-through rate**, i.e., ratio of users that click on a specific link to the total number of users who view the page (a proportion that goes from 0 to 1).

We have already processed the data for you. Firstly, we load the Crazy Egg data from the web.

In [ ]:
click_through <- 
    read_csv("data/click_through.csv") %>% 
    select(webpage, adjusted_clicks, target_clicks)

head(click_through)

**Question 2.2**
<br>{points: 1}

The `adjusted_clicks` in the data frame `click_through` are the total clicks we will use to compute the click-through rate by treatment, where `target_clicks` are what we could define as **“successes”**. Compute the corresponding click-through rate by row by dividing `target_clicks` over `adjusted_clicks`. Add it as a new column in the data frame called `click_rate`. Then, reorder the experimental treatments (i.e., factor levels) in descending order by click-through rate.

*Use the scaffolding below.*

In [ ]:
# click_through <- 
#     ... %>% 
#     mutate(click_rate = ...) %>% 
#     mutate(webpage = fct_reorder(..., desc(...)))

# your code here
fail() # No Answer - remove if you provide an answer

click_through
levels(click_through$webpage)

In [ ]:
test_2.2()

**Question 2.3**
<br>{points: 1}

The sampled click-through rates in the data frame `click_through` are estimates of population proportions. Hence, it is possible to obtain confidence intervals by relying on the Central Limit Theorem. Obtain the 95% confidence interval for each population click rate and store the lower and upper bounds in two new columns `click_through`: `lower_ci` and `upper_ci`.

In [ ]:
# click_through <- 
#     click_through %>% 
#     ...(lower_ci = ...,
#            upper_ci = ...)

# your code here
fail() # No Answer - remove if you provide an answer

click_through

In [ ]:
test_2.3()

**Question 2.4**
<br>{points: 1}

Let's create an effective visualization for the point estimate click_rate and the confidence intervals you obtained above. The `ggplot()` object's name shoud be `CIs_click_through_rates`.

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*

In [ ]:
# Plotting click-through rates as points with 95% confidence intervals.
# CIs_click_through_rates <- 
#   click_through %>% 
#   ggplot(aes(..., ...)) +
#   ...() +
#   geom_errorbar(aes(ymin = ..., ymax = ...), width = 0.1) +
#   theme(
#     text = element_text(size = 22),
#     plot.title = element_text(face = "bold"),
#     axis.title = element_text(face = "bold")
#   ) +
#   ggtitle(...) +
#   xlab(...) +
#   ylab(...)

# your code here
fail() # No Answer - remove if you provide an answer

CIs_click_through_rates

In [ ]:
test_2.4()

**Question 2.5**
<br>{points: 1}

Based on the findings in the plot `CIs_click_through_rates`, what can we statistically conclude from these confidence intervals?

**A.** We can see that treatment **Connect** has the largest click-through rate among the five treatments. It is statistically larger than the control treatment **Interact**.

**B.** We cannot state that treatment **Connect** is statistically larger than treatment **Services** since their confidence intervals overlap. However, we can state that these two treatments are statistically larger than the control treatment **Interact** and **Learn** given that their corresponding confidence intervals do not overlap.

*Assign your answer to an object called `answer2.5`. Your answer should be one of `"A"` or `"B"` surrounded by quotes.*

In [ ]:
# answer2.5 <- 

# your code here
fail() # No Answer - remove if you provide an answer

In [ ]:
test_2.5()

**Question 2.6**
<br>{points: 1}

Recall that the click-through rates by treatment are proportions that go from 0 to 1. We want to compare whether the rate of a given treatment is larger than the rate corresponding to another one. Suppose we rely on the Central Limit Theorem and assume that our sample sizes are large enough. What is the specific analysis we need to perform?

**A.** One-sample $z$-test. 

**B.** One-sample $t$-test.

**C.** Two-sample $z$-test.

**D.** Two-sample $t$-test.

**E.** Two-way ANOVA.

*Assign your answer to an object called `answer2.6`. Your answer should be one of `"A"`, `"B"`, `"C"`, `"D"`, or `"E"` surrounded by quotes.*

In [ ]:
# answer2.6 <- 

# your code here
fail() # No Answer - remove if you provide an answer

In [ ]:
test_2.6()

**Question 2.7**
<br>{points: 1}

Let $p_A$ and $p_B$ be the click-through rates of two given treatments **A** and **B**, respectively. Suppose you want to assess whether the click-through rate of treatment **A** is larger than the one corresponding to treatment **B**. What is the set of hypotheses we are testing in this case?

**A.** $H_0: p_A = p_B$ vs. $H_1: p_A > p_B$

**B.** $H_0: p_A > p_B$ vs. $H_1: p_A < p_B$

**C.** $H_0: p_A = p_B$ vs. $H_1: p_A \neq p_B$

**D.** $H_0: p_A = p_B$ vs. $H_1: p_A < p_B$

*Assign your answer to an object called `answer2.7`. Your answer should be one of `"A"`, `"B"`, `"C"`, or `"D"` surrounded by quotes.*

In [ ]:
# answer2.7 <- 

# your code here
fail() # No Answer - remove if you provide an answer

In [ ]:
test_2.7()

**Question 2.7**
<br>{points: 1}

Perform pairwise frequentist hypothesis test analyses to assess statistically significant differences between all the experimental treatments. This will require control for multiple comparisons. You can use the Bonferroni correction along with the function `pairwise.prop.test()`. Create an object named `pairwise_comparisons`. 

> **Heads-up:** Given the answer in **Question 2.6**, rows in `pairwise_comparisons` will correspond to treatment **A** and columns to treatment **B**.

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*

In [ ]:
# Assigning numbers of "successes" from data frame `click_through`
# successes <- click_through$...

# Assigning numbers of "trials" from data frame `click_through`
# trials <- click_through$...

# Putting labels on vector cells
# names(successes) <- click_through$webpage
# names(trials) <- click_through$webpage

# pairwise_comparisons <- pairwise.prop.test(x = ...,
#   n = ....,
#   p.adjust.method = ..., 
#   alternative = ...,
# )

# your code here
fail() # No Answer - remove if you provide an answer

In [ ]:
test_2.7()

**Question 2.8**
<br>{points: 1}

Based on your results in **Question 2.8**, using $\alpha = 0.05$, indicate what experimental treatments have a significantly larger click-through rate than the control **Interact**.

**A.** Connect.

**B.** Learn.

**C.** Help.

**D.** Services.

*Assign your answers to the object `answer2.8`. Your answer has to be a single string indicating the correct treatment labels **in alphabetical order** and surrounded by quotes (e.g., `"ABCD"` indicates you are selecting the four options).*

In [ ]:
# answer2.8 <- 

# your code here
fail() # No Answer - remove if you provide an answer

In [ ]:
test_2.8()